# 6.3 scikit-learn → ONNX — Deep Dive

## Table of Contents
1. [The sklearn → ONNX Pipeline](#section-1)
2. [Pipeline Linearization](#section-2)
3. [Tree Model Encoding as Sum-of-Products](#section-3)
4. [Linear Model Encoding](#section-4)
5. [The `initial_types` Contract](#section-5)
6. [Converting a LogisticRegression Model](#section-6)
7. [Converting a RandomForest Model](#section-7)
8. [Converting a Full Pipeline](#section-8)
9. [Parity Checking](#section-9)
10. [Supported Operators](#section-10)
11. [Key Takeaways](#section-11)

<a id='section-1'></a>
## Section 1: The sklearn → ONNX Pipeline

Converting scikit-learn models to ONNX is fundamentally different from
converting deep learning frameworks. scikit-learn models are **Python
objects with fitted parameters** — there is no computation graph to
translate. Instead, `skl2onnx` must **reimplement** each sklearn estimator's
prediction logic as an ONNX subgraph.

```
┌──────────────────────────────────────────────────────────────────────────┐
│                   sklearn → ONNX PIPELINE                               │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│   sklearn Estimator          skl2onnx             ONNX Graph            │
│  ┌──────────────────┐      ┌──────────────┐     ┌────────────────┐     │
│  │ Python object     │      │ 1. Parse     │     │ NodeProto      │     │
│  │ with fitted       │─────▶│    topology   │────▶│ objects using  │     │
│  │ parameters        │      │ 2. Extract   │     │ ai.onnx.ml     │     │
│  │ (.coef_, .tree_)  │      │    params    │     │ domain ops     │     │
│  └──────────────────┘      │ 3. Build ONNX│     └────────────────┘     │
│                             │    subgraph  │            │               │
│                             └──────────────┘            ▼               │
│                                                   ┌──────────┐         │
│  KEY INSIGHT: skl2onnx does NOT execute Python     │ ORT      │         │
│  prediction code at inference time. It builds a    │ Inference│         │
│  static ONNX graph that reimplements the same      │ Session  │         │
│  mathematical operations.                          └──────────┘         │
└──────────────────────────────────────────────────────────────────────────┘
```

### Why This Matters

Since skl2onnx builds a **new implementation** rather than translating an
existing graph, the converter must:

1. Understand the internal structure of each sklearn estimator
2. Extract all fitted parameters ($w$, $b$, tree structures, etc.)
3. Reconstruct the prediction logic using ONNX operators
4. Handle sklearn's `predict()` AND `predict_proba()` in a single graph

This means skl2onnx must be explicitly updated whenever sklearn adds
new estimators or changes internal parameter formats.

<a id='section-2'></a>
## Section 2: Pipeline Linearization

### How sklearn Pipelines Work

A sklearn `Pipeline` chains transformers and a final estimator:

$$f_{\text{pipeline}}(X) = f_n \circ f_{n-1} \circ \cdots \circ f_1(X)$$

where each $f_i$ is either a **transformer** (StandardScaler, PCA, etc.)
or the final **estimator** (classifier/regressor).

### ONNX Linearization

When skl2onnx converts a Pipeline, each step becomes a **subgraph** of
ONNX nodes, connected sequentially:

```
┌─────────────────────────────────────────────────────────────────┐
│              PIPELINE → ONNX LINEARIZATION                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  sklearn Pipeline:                                              │
│  ┌───────────┐   ┌───────────┐   ┌────────────────┐           │
│  │ Standard  │──▶│    PCA    │──▶│   Logistic     │           │
│  │ Scaler    │   │           │   │   Regression   │           │
│  └───────────┘   └───────────┘   └────────────────┘           │
│                                                                 │
│  ONNX Graph:                                                    │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ Input                                                    │   │
│  │   │                                                      │   │
│  │   ▼  Scaler subgraph                                    │   │
│  │ [Sub(mean)] → [Div(std)]                                │   │
│  │   │                                                      │   │
│  │   ▼  PCA subgraph                                       │   │
│  │ [Sub(pca_mean)] → [MatMul(components)]                  │   │
│  │   │                                                      │   │
│  │   ▼  LogisticRegression subgraph                        │   │
│  │ [MatMul(coef)] → [Add(intercept)] → [Sigmoid]           │   │
│  │   │                                                      │   │
│  │   ▼                                                      │   │
│  │ Output (label + probabilities)                           │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  Benefit: ONE ONNX file = preprocessing + model                │
│  No training/serving skew — same transforms guaranteed         │
└─────────────────────────────────────────────────────────────────┘
```

### Mathematical View

For a pipeline with StandardScaler ($\mu$, $\sigma$), PCA ($W_{\text{pca}}$,
$\mu_{\text{pca}}$), and LogisticRegression ($w$, $b$):

$$\hat{y} = \sigma\left(w^T \cdot W_{\text{pca}}^T \cdot \frac{X - \mu}{\sigma_{\text{scale}}} + b'\right)$$

where all intermediate bias adjustments are folded into $b'$. The ONNX graph
preserves this chain but uses explicit nodes for each operation.

<a id='section-3'></a>
## Section 3: Tree Model Encoding as Sum-of-Products

### Decision Tree Prediction

A decision tree partitions the feature space into axis-aligned regions.
Each leaf $l$ has a predicted value $v_l$. The prediction function is:

$$f(x) = \sum_{l \in \text{leaves}} v_l \cdot \prod_{(j, t, d) \in \text{path}(l)} \mathbb{1}[x_j \leq t]^{\mathbb{1}[d=\text{left}]} \cdot \mathbb{1}[x_j > t]^{\mathbb{1}[d=\text{right}]}$$

In simpler terms: traverse the tree from root to leaf following split
conditions, then return the leaf's value. Only one leaf is active per
input — the products ensure exactly one term in the sum is non-zero.

### ONNX Representation

ONNX provides specialized operators in the `ai.onnx.ml` domain for tree
ensembles:

- `TreeEnsembleClassifier` — for classification trees/forests
- `TreeEnsembleRegressor` — for regression trees/forests

These operators encode the **entire tree structure** as operator attributes:

```
┌─────────────────────────────────────────────────────────────┐
│          TREE ENCODING IN ONNX                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Decision Tree:              ONNX Attributes:               │
│                                                             │
│        [x₀ ≤ 0.5]           nodes_featureids = [0, 1, -, -]│
│        /         \           nodes_values     = [0.5, 0.3,  │
│   [x₁ ≤ 0.3]   leaf(C)                         -, -]      │
│    /       \                 nodes_modes       = [BRANCH_LEQ│
│ leaf(A)  leaf(B)                                 BRANCH_LEQ,│
│                                                  LEAF, LEAF]│
│                              nodes_treeids     = [0, 0, 0, 0│
│                              nodes_nodeids     = [0, 1, 2, 3│
│                              nodes_truenodeids = [1, 2, 0, 0│
│                              nodes_falsenodeids= [3, 3, 0, 0│
│                                                             │
│  The entire tree is flattened into parallel arrays.          │
│  Traversal at inference uses array indexing, not pointers.   │
└─────────────────────────────────────────────────────────────┘
```

### Random Forest Encoding

For an ensemble of $T$ trees, a Random Forest's prediction is:

$$\hat{y}_{\text{RF}}(x) = \frac{1}{T} \sum_{t=1}^{T} f_t(x)$$

where each $f_t$ is a decision tree. In ONNX, all trees are packed into a
single `TreeEnsembleClassifier` node with `nodes_treeids` distinguishing
which nodes belong to which tree.

### Efficiency Consideration

The flattened array representation enables efficient inference:
- No pointer chasing (cache-friendly)
- Vectorizable across samples
- Single operator call for the entire ensemble

<a id='section-4'></a>
## Section 4: Linear Model Encoding

### The Mathematical Model

Linear models in sklearn compute:

$$\hat{y} = \sigma(Xw + b)$$

where $\sigma$ is the activation function:
- **LinearRegression**: $\sigma = \text{identity}$
- **LogisticRegression**: $\sigma = \text{sigmoid}$ (binary) or $\sigma = \text{softmax}$ (multinomial)
- **Ridge**: $\sigma = \text{identity}$ (regularization only affects training)

### ONNX Mapping

The conversion maps the linear model to standard ONNX tensor operations:

```
sklearn LinearRegression:          ONNX Graph:
─────────────────────────         ─────────────

y = X @ coef_ + intercept_        Input(X)
                                     │
                                  MatMul(X, coef_)  ← coef_ as initializer
                                     │
                                  Add(_, intercept_) ← intercept_ as initializer
                                     │
                                  Output(y)


sklearn LogisticRegression:        ONNX Graph:
──────────────────────────        ─────────────

z = X @ coef_.T + intercept_      Input(X)
p = sigmoid(z)  [binary]             │
p = softmax(z)  [multinomial]     LinearClassifier  ← ai.onnx.ml domain
label = argmax(p)                    │         │
                                  label    probabilities
```

### ONNX ML Domain Operators

For classifiers, skl2onnx often uses the `ai.onnx.ml` domain's
`LinearClassifier` operator, which combines the linear transform
and activation in a single node:

| sklearn Model | ONNX Operator(s) | Domain |
|---------------|-------------------|--------|
| LinearRegression | MatMul + Add | ai.onnx (standard) |
| LogisticRegression | LinearClassifier | ai.onnx.ml |
| Ridge | MatMul + Add | ai.onnx (standard) |
| SGDClassifier | LinearClassifier | ai.onnx.ml |

<a id='section-5'></a>
## Section 5: The `initial_types` Contract

### Why `initial_types` is Required

Unlike deep learning frameworks where the model's graph carries type
information, sklearn estimators don't know their input type at export
time. You must explicitly declare it.

### The Declaration Format

```python
initial_types = [("input_name", TypeObject([shape]))]
```

### Available Type Objects

| Type | Python Class | When to Use |
|------|-------------|------------|
| float32 | `FloatTensorType([None, n])` | **Default choice** — most ONNX ops optimize for fp32 |
| float64 | `DoubleTensorType([None, n])` | When sklearn trained on float64 and precision matters |
| int64 | `Int64TensorType([None, n])` | Integer features (categorical indices, counts) |
| string | `StringTensorType([None, n])` | Text features (with TF-IDF, CountVectorizer) |

### Shape Convention

The shape uses `None` for the batch dimension (variable number of samples)
and the exact feature count for the second dimension:

$$\text{shape} = [\underbrace{\texttt{None}}_{\text{batch}},\; \underbrace{n_{\text{features}}}_{\text{from training data}}]$$

### Common Mistake: dtype Mismatch

If sklearn trained on float64 data but you declare `FloatTensorType`
(float32), the conversion will cast internally. This usually works but
can introduce small numerical differences. Best practice:

```python
X_train = X_train.astype(np.float32)  # cast BEFORE training
model.fit(X_train, y_train)
initial_types = [("X", FloatTensorType([None, X_train.shape[1]]))]  # match
```

In [ ]:
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import onnx
from onnx import checker

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType, DoubleTensorType, Int64TensorType
    import onnxruntime as ort
    HAS_SKL2ONNX = True
    print("skl2onnx and onnxruntime available!")
except ImportError as e:
    HAS_SKL2ONNX = False
    print(f"Missing dependency: {e}")
    print("Install with: pip install skl2onnx onnxruntime")

<a id='section-6'></a>
## Section 6: Converting a LogisticRegression Model

LogisticRegression is the simplest sklearn model to understand in ONNX
form — it maps directly to a `LinearClassifier` node.

### The Math

For binary classification with features $x \in \mathbb{R}^d$:

$$P(y=1|x) = \frac{1}{1 + e^{-(w^T x + b)}}$$

For multinomial ($K$ classes):

$$P(y=k|x) = \frac{e^{w_k^T x + b_k}}{\sum_{j=1}^{K} e^{w_j^T x + b_j}}$$

The ONNX graph packages $w \in \mathbb{R}^{K \times d}$ and $b \in \mathbb{R}^K$
as initializers inside a `LinearClassifier` node.

In [ ]:
if HAS_SKL2ONNX:
    X, y = make_classification(
        n_samples=300, n_features=8, n_informative=5,
        n_classes=3, random_state=42
    )
    X = X.astype(np.float32)

    clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
    clf.fit(X, y)

    initial_type = [("X", FloatTensorType([None, 8]))]
    onnx_model = convert_sklearn(clf, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_model)

    print(f"LogisticRegression converted!")
    print(f"  Nodes: {len(onnx_model.graph.node)}")
    print(f"  Ops: {[n.op_type for n in onnx_model.graph.node]}")
    print(f"  Outputs: {[o.name for o in onnx_model.graph.output]}")
    print(f"  Coef shape: {clf.coef_.shape}")
    print(f"  Intercept: {clf.intercept_}")
else:
    print("Skipped — skl2onnx not available.")

<a id='section-7'></a>
## Section 7: Converting a RandomForest Model

RandomForest models are more complex — the converter must encode every
tree's structure into a single `TreeEnsembleClassifier` node.

### What Gets Encoded

For a forest with $T$ trees, each with up to $N$ nodes:

| Attribute | Content | Size |
|-----------|---------|------|
| `nodes_treeids` | Which tree each node belongs to | $T \times N$ |
| `nodes_nodeids` | Node index within its tree | $T \times N$ |
| `nodes_featureids` | Feature index for split | $T \times N$ |
| `nodes_values` | Split threshold value | $T \times N$ |
| `nodes_modes` | BRANCH_LEQ or LEAF | $T \times N$ |
| `class_weights` | Class probabilities at each leaf | Leaves $\times K$ |

In [ ]:
if HAS_SKL2ONNX:
    X_rf, y_rf = make_classification(
        n_samples=500, n_features=12, n_informative=8,
        n_classes=2, random_state=0
    )
    X_rf = X_rf.astype(np.float32)

    rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=0)
    rf.fit(X_rf, y_rf)

    initial_type = [("X", FloatTensorType([None, 12]))]
    onnx_rf = convert_sklearn(rf, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_rf)

    total_nodes = sum(tree.tree_.node_count for tree in rf.estimators_)
    total_leaves = sum(
        (tree.tree_.children_left == -1).sum() for tree in rf.estimators_
    )

    print(f"RandomForest converted!")
    print(f"  Trees: {rf.n_estimators}")
    print(f"  Total tree nodes: {total_nodes:,}")
    print(f"  Total leaves: {total_leaves:,}")
    print(f"  ONNX graph nodes: {len(onnx_rf.graph.node)}")
    print(f"  ONNX ops: {[n.op_type for n in onnx_rf.graph.node]}")

    serialized = onnx_rf.SerializeToString()
    print(f"  Serialized size: {len(serialized):,} bytes ({len(serialized)/1024:.1f} KB)")
else:
    print("Skipped — skl2onnx not available.")

<a id='section-8'></a>
## Section 8: Converting a Full Pipeline

The real power of skl2onnx shines when converting **pipelines** — the
preprocessing and model are packaged into a single ONNX file, eliminating
training/serving skew.

### Why This Matters

```
WITHOUT Pipeline-in-ONNX:                WITH Pipeline-in-ONNX:
────────────────────────                ───────────────────────

Training:                               Training:
  scaler.fit(X) → pca.fit(X_s) → ...     pipe.fit(X, y)

Serving:                                Serving:
  X_s = scaler.transform(X)  ← BUG?      ort.run(X)  ← ONE CALL
  X_p = pca.transform(X_s)  ← RISK                     No skew
  y = model.predict(X_p)     ← SKEW                    possible

The serving code must EXACTLY replicate the training
preprocessing — a common source of production bugs.
```

### Pipeline with ColumnTransformer

For real-world data with mixed feature types, `ColumnTransformer`
applies different preprocessing to different columns. skl2onnx handles
this by building parallel subgraphs that merge via `Concat`.

In [ ]:
if HAS_SKL2ONNX:
    X_pipe, y_pipe = make_classification(
        n_samples=400, n_features=15, n_informative=10,
        n_classes=3, random_state=1
    )
    X_pipe = X_pipe.astype(np.float32)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=8)),
        ("clf", GradientBoostingClassifier(
            n_estimators=30, max_depth=3, random_state=0
        )),
    ])
    pipe.fit(X_pipe, y_pipe)

    initial_type = [("X", FloatTensorType([None, 15]))]
    onnx_pipe = convert_sklearn(pipe, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_pipe)

    all_ops = [n.op_type for n in onnx_pipe.graph.node]
    unique_ops = sorted(set(all_ops))

    print(f"Pipeline converted!")
    print(f"  Steps: {[name for name, _ in pipe.steps]}")
    print(f"  ONNX nodes: {len(all_ops)}")
    print(f"  Unique ops: {unique_ops}")
    print(f"  Outputs: {[o.name for o in onnx_pipe.graph.output]}")
else:
    print("Skipped — skl2onnx not available.")

<a id='section-9'></a>
## Section 9: Parity Checking

### The Parity Test for sklearn Models

sklearn models produce two types of output that must be verified:

1. **`predict(X)`** — class labels (classifiers) or continuous values (regressors)
2. **`predict_proba(X)`** — class probability estimates (classifiers only)

The ONNX graph typically produces **both** as separate outputs:

```
ONNX Graph Outputs:
──────────────────
  output[0]  →  predicted labels  (int64 array)
  output[1]  →  probability map   (list of {class: prob} dicts)
                 or probability array
```

### Tolerance for sklearn Models

For tree-based models, predictions should be **exactly equal** (tree
traversal is deterministic). For linear models with floating-point
arithmetic, small differences are expected:

$$\max_i |p^{\text{sklearn}}_i - p^{\text{ORT}}_i| < \epsilon \approx 10^{-6}$$

In [ ]:
if HAS_SKL2ONNX:
    print("=" * 60)
    print("PARITY CHECK: LogisticRegression")
    print("=" * 60)

    sess_lr = ort.InferenceSession(
        onnx_model.SerializeToString(), providers=["CPUExecutionProvider"]
    )

    X_test = X[:20]
    sk_labels = clf.predict(X_test)
    sk_proba = clf.predict_proba(X_test)

    feeds = {sess_lr.get_inputs()[0].name: X_test}
    ort_outputs = sess_lr.run(None, feeds)
    ort_labels = ort_outputs[0]

    label_match = np.array_equal(sk_labels, ort_labels)
    print(f"  Labels match: {label_match}")
    print(f"  First 10 sklearn labels: {sk_labels[:10]}")
    print(f"  First 10 ORT labels:     {ort_labels[:10]}")

    ort_proba = None
    for out in ort_outputs:
        if isinstance(out, np.ndarray) and out.ndim == 2 and out.shape[0] == len(X_test):
            ort_proba = out
            break

    if ort_proba is not None:
        proba_diff = np.abs(sk_proba - ort_proba).max()
        print(f"  Max probability diff: {proba_diff:.2e}")
        print(f"  Parity OK: {proba_diff < 1e-5}")
    else:
        print("  Could not extract probability output for comparison.")

    print()
    print("=" * 60)
    print("PARITY CHECK: RandomForest")
    print("=" * 60)

    sess_rf = ort.InferenceSession(
        onnx_rf.SerializeToString(), providers=["CPUExecutionProvider"]
    )

    X_test_rf = X_rf[:20]
    sk_labels_rf = rf.predict(X_test_rf)
    sk_proba_rf = rf.predict_proba(X_test_rf)

    feeds_rf = {sess_rf.get_inputs()[0].name: X_test_rf}
    ort_outputs_rf = sess_rf.run(None, feeds_rf)
    ort_labels_rf = ort_outputs_rf[0]

    label_match_rf = np.array_equal(sk_labels_rf, ort_labels_rf)
    print(f"  Labels match: {label_match_rf}")

    ort_proba_rf = None
    for out in ort_outputs_rf:
        if isinstance(out, np.ndarray) and out.ndim == 2 and out.shape[0] == len(X_test_rf):
            ort_proba_rf = out
            break

    if ort_proba_rf is not None:
        proba_diff_rf = np.abs(sk_proba_rf - ort_proba_rf).max()
        print(f"  Max probability diff: {proba_diff_rf:.2e}")
        print(f"  Parity OK: {proba_diff_rf < 1e-5}")
else:
    print("Skipped — skl2onnx not available.")

In [ ]:
if HAS_SKL2ONNX:
    print("=" * 60)
    print("PARITY CHECK: Full Pipeline (Scaler → PCA → GradientBoosting)")
    print("=" * 60)

    sess_pipe = ort.InferenceSession(
        onnx_pipe.SerializeToString(), providers=["CPUExecutionProvider"]
    )

    X_test_pipe = X_pipe[:30]
    sk_labels_pipe = pipe.predict(X_test_pipe)

    feeds_pipe = {sess_pipe.get_inputs()[0].name: X_test_pipe}
    ort_outputs_pipe = sess_pipe.run(None, feeds_pipe)
    ort_labels_pipe = ort_outputs_pipe[0]

    label_match_pipe = np.array_equal(sk_labels_pipe, ort_labels_pipe)
    print(f"  Labels match: {label_match_pipe}")
    print(f"  Accuracy (sklearn): {(sk_labels_pipe == y_pipe[:30]).mean():.2%}")
    print(f"  Accuracy (ORT):     {(ort_labels_pipe == y_pipe[:30]).mean():.2%}")
else:
    print("Skipped — skl2onnx not available.")

### Inspecting ONNX Graph Structure

Understanding what skl2onnx generates helps debug conversion issues.
Let's examine the internal graph structure of converted models.

For a `LogisticRegression`, the graph typically contains:

$$\text{Input}(X) \xrightarrow{\text{LinearClassifier}} \begin{pmatrix} \text{labels} \\ \text{probabilities} \end{pmatrix}$$

For a `Pipeline`, the graph chains multiple subgraphs:

$$\text{Input}(X) \xrightarrow{\text{Scaler}} X' \xrightarrow{\text{PCA}} X'' \xrightarrow{\text{Classifier}} (\text{labels}, \text{proba})$$

In [ ]:
if HAS_SKL2ONNX:
    def inspect_onnx_graph(model_proto, name="Model"):
        """Display detailed ONNX graph structure."""
        print(f"\n{'=' * 55}")
        print(f"Graph Inspection: {name}")
        print(f"{'=' * 55}")

        print(f"\nInputs:")
        for inp in model_proto.graph.input:
            shape = inp.type.tensor_type.shape
            if shape:
                dims = [d.dim_param or str(d.dim_value) for d in shape.dim]
                print(f"  {inp.name}: [{', '.join(dims)}]")

        print(f"\nNodes ({len(model_proto.graph.node)}):")
        for i, node in enumerate(model_proto.graph.node):
            in_names = ', '.join(node.input[:3])
            out_names = ', '.join(node.output[:2])
            domain = f" ({node.domain})" if node.domain else ""
            print(f"  [{i}] {node.op_type}{domain}: [{in_names}] → [{out_names}]")

        print(f"\nOutputs:")
        for out in model_proto.graph.output:
            print(f"  {out.name}")

        print(f"\nInitializers: {len(model_proto.graph.initializer)}")
        for init in model_proto.graph.initializer:
            print(f"  {init.name}: shape={list(init.dims)}, dtype={init.data_type}")

    inspect_onnx_graph(onnx_model, "LogisticRegression")
else:
    print("Skipped — skl2onnx not available.")

In [ ]:
if HAS_SKL2ONNX:
    inspect_onnx_graph(onnx_pipe, "Pipeline (Scaler + PCA + GradientBoosting)")
else:
    print("Skipped — skl2onnx not available.")

### Custom Converter Registration

If you use a custom sklearn-compatible transformer, you need to register
a custom converter with skl2onnx. The registration API:

```python
from skl2onnx import update_registered_converter
from skl2onnx.common.data_types import FloatTensorType

def my_shape_calculator(operator):
    """Declare output shapes."""
    operator.outputs[0].type = FloatTensorType([None, n_out])

def my_converter(scope, operator, container):
    """Build ONNX subgraph for the custom transformer."""
    X = operator.inputs[0]
    out = operator.outputs[0]
    # Add ONNX nodes using container.add_node(...)

update_registered_converter(
    MyCustomTransformer,       # your class
    'MyCustomTransformer',     # alias
    my_shape_calculator,
    my_converter
)
```

### When Custom Registration is Needed

| Scenario | Solution |
|----------|----------|
| Custom FunctionTransformer | Register converter |
| Third-party sklearn-compatible lib | Check if skl2onnx already supports it |
| Wrapped model (e.g., CalibratedClassifierCV) | May need manual converter |
| Custom pipeline step with side effects | Redesign as pure function |

<a id='section-10'></a>
## Section 10: Supported Operators

### Comprehensive Operator Support Table

skl2onnx supports a large subset of sklearn estimators. Here is a
representative catalog organized by category:

| Category | Estimator / Transformer | ONNX Op(s) Used |
|----------|------------------------|----------------|
| **Preprocessing** | StandardScaler | Sub + Div (or Scaler) |
| | MinMaxScaler | Sub + Div + Mul + Add |
| | MaxAbsScaler | Div |
| | RobustScaler | Sub + Div |
| | OneHotEncoder | OneHotEncoder (ai.onnx.ml) |
| | LabelEncoder | LabelEncoder (ai.onnx.ml) |
| | Binarizer | Binarizer (ai.onnx.ml) |
| **Decomposition** | PCA | Sub + MatMul |
| | TruncatedSVD | MatMul |
| **Linear Models** | LinearRegression | MatMul + Add |
| | LogisticRegression | LinearClassifier (ai.onnx.ml) |
| | Ridge / Lasso | MatMul + Add |
| | SGDClassifier | LinearClassifier |
| **Tree Models** | DecisionTreeClassifier | TreeEnsembleClassifier |
| | DecisionTreeRegressor | TreeEnsembleRegressor |
| | RandomForestClassifier | TreeEnsembleClassifier |
| | GradientBoostingClassifier | TreeEnsembleClassifier |
| | ExtraTreesClassifier | TreeEnsembleClassifier |
| **SVM** | SVC | SVMClassifier (ai.onnx.ml) |
| | SVR | SVMRegressor (ai.onnx.ml) |
| **Neighbors** | KNeighborsClassifier | Custom subgraph |
| **Pipeline** | Pipeline | Sequential subgraphs |
| | ColumnTransformer | Parallel subgraphs + Concat |
| | FeatureUnion | Parallel subgraphs + Concat |

### Unsupported or Partially Supported

Some sklearn components are difficult to represent in ONNX:

- **Custom transformers** — need custom converter registration
- **Calibrated classifiers** — partial support
- **Stacking/Voting** — may require manual composition
- **Feature selectors** — some missing

<a id='section-11'></a>
## Section 11: Key Takeaways

### Conversion Checklist

```
┌─────────────────────────────────────────────────────────────────┐
│           sklearn → ONNX CONVERSION CHECKLIST                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  □  1. Train on float32 data (match ONNX precision)            │
│  □  2. Declare initial_types with correct shape and dtype      │
│  □  3. Use convert_sklearn() with target_opset                 │
│  □  4. Validate with onnx.checker.check_model()                │
│  □  5. Verify predict() parity (labels must match exactly)     │
│  □  6. Verify predict_proba() parity (diff < 1e-6)            │
│  □  7. Export Pipelines as single ONNX file                    │
│  □  8. Test with variable batch sizes                          │
│  □  9. Include parity test in CI pipeline                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Core Concepts Summary

| Concept | Key Insight |
|---------|------------|
| **Reimplementation** | skl2onnx builds a NEW ONNX graph that reimplements sklearn's prediction logic — it does NOT trace Python code |
| **Pipeline Linearization** | $f_{\text{pipe}} = f_n \circ \cdots \circ f_1$ maps to sequential ONNX subgraphs; eliminates training/serving skew |
| **Tree Encoding** | Trees are flattened into parallel arrays (feature ids, thresholds, leaf values) in a single TreeEnsemble node |
| **Linear Encoding** | $\hat{y} = \sigma(Xw + b)$ maps to MatMul + Add + activation, or a single LinearClassifier node |
| **initial_types** | You must declare input shape and dtype; use float32 by default for best ONNX compatibility |
| **ai.onnx.ml Domain** | sklearn converters use ML-specific ONNX operators (TreeEnsemble, LinearClassifier, Scaler) |